# Baseline ResNet50

Notebook ini berfungsi sebagai orchestrator training dan evaluasi model.`

In [ ]:
import tensorflow as tf

from configs.config import CONFIG
from src.data.dataset import build_dataset
from src.data.split import split_dataset
from src.models.resnet50 import build_resnet50_model
from src.training.callbacks import build_callbacks
from src.training.trainer import ModelTrainer
from src.utils.seed import set_seed

set_seed(CONFIG.SEED)

split = split_dataset(CONFIG.DATA_DIR, test_size=CONFIG.TEST_SPLIT, val_size=CONFIG.VALIDATION_SPLIT, random_state=CONFIG.SEED)
train_paths, train_labels = split['train']
val_paths, val_labels = split['val']
test_paths, test_labels = split['test']

train_ds = build_dataset(train_paths, train_labels, batch_size=CONFIG.BATCH_SIZE, shuffle=True)
val_ds = build_dataset(val_paths, val_labels, batch_size=CONFIG.BATCH_SIZE, shuffle=False)
test_ds = build_dataset(test_paths, test_labels, batch_size=CONFIG.BATCH_SIZE, shuffle=False)

model = build_resnet50_model(
    input_shape=(224, 224, 3),
    num_classes=len(CONFIG.CLASS_NAMES),
    freeze_base_model=CONFIG.FREEZE_BASE_MODEL,
    dropout_rate=CONFIG.DROPOUT_RATE,
)

trainer = ModelTrainer(model, train_ds, val_ds, callbacks=build_callbacks())
history = trainer.fit(epochs=CONFIG.EPOCHS)
trainer.save_history(history, 'results/history.json')
trainer.save_model('checkpoints/final_model.keras')
print('Training complete')